In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/ventas_raw.csv')
df

,id_venta,numero_linea,fecha_venta,id_cliente,id_producto,id_tienda,nombre_tienda,ciudad,empresa,nombre_producto,...,precio_unitario,descuento,genero_cliente,tipo_cliente,talla,metodo_pago,canal,temporada,tipo_prenda,departamento
0,V003565,1,2024-02-04,C00909,P0003,T007,KOAJ San Pedro Plaza,Neiva,KOAJ,Jean mom,...,"179.000,00",0.00,Hombre,Local,40,Tarjeta crédito,Tienda física,Temporada regular,Pantalón,Huila
1,V002111,2,2023-01-01,C00348,P0008,T040,KOAJ Andino,Bogotá,KOAJ,Chaqueta bomber,...,"211000,00",0.00,Hombre,Local,M,Efectivo,Tienda física,Regreso a clases,Chaqueta,Bogotá D.C.
2,V002185,1,2023-03-31,C00804,P0009,T003,KOAJ Ocean Mall,Santa Marta,KOAJ,Camisa casual,...,"128000,00",0.00,Mujer,Turista,XXL,Efectivo,Tienda física,Día del Padre,Camisa,Magdalena
3,V002828,1,2024-05-13,C00951,P0014,T034,KOAJ Viva Barranquilla,Barranquilla,KOAJ,Jogger,...,"133000,00",NaN,Hombre,Local,38,Tarjeta crédito,Tienda física,Black Friday,Pantalón,Atlántico
4,V001693,1,2023-11-14,C01804,P0007,T030,KOAJ Cacique,Bucaramanga,KOAJ,Gorra,...,"61000,00",0.00,Mujer,Local,XL,Tarjeta crédito,Tienda física,Temporada regular,Accesorio,Santander
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,V003749,1,2024-10-19,C01395,P0004,T005,KOAJ Viva Sincelejo,Sincelejo,KOAJ,Jean skinny,...,"180000,00",0.00,Mujer,Local,38,Tarjeta crédito,Tienda física,Temporada de diciembre,Pantalón,Sucre
5996,V002277,2,2025-12-04,C02228,P0009,T019,KOAJ Primavera Urbana,Villavicencio,KOAJ,Camisa casual,...,"130.000,00",0.00,Mujer,Local,S,Tarjeta crédito,Tienda física,Temporada de diciembre,Camisa,Meta
5997,V001587,1,2025-07-11,C01323,P0018,T018,KOAJ Mall Plaza,Cartagena,KOAJ,Camiseta básica,...,77000,0.05,Mujer,Local,M,Tarjeta débito,Tienda física,Temporada regular,Camiseta,Bolívar
5998,V001576,1,2024-01-15,C02669,P0021,T018,KOAJ Mall Plaza,Cartagena,KOAJ,Polo,...,"126000,00",0.05,Mujer,Local,L,Efectivo,Tienda física,Temporada regular,Polo,Bolívar


In [2]:
# Normalizar empresa

print(f"Antes: {df['empresa'].value_counts()}")
df['empresa'] = df['empresa'].str.upper().str.strip()
print(f"\nDespués: {df['empresa'].value_counts()}")

Antes: empresa
KOAJ     5900
KOAJ       70
koaj       30
Name: count, dtype: int64

Después: empresa
KOAJ    6000
Name: count, dtype: int64


In [3]:
# Limpiar precios -_-

mask = df['precio_unitario'].astype(str).str.contains(',', na=False)

df.loc[mask, 'precio_unitario'] = df.loc[mask, 'precio_unitario'].str.replace('.', '', regex=False)
df['precio_unitario'] = df['precio_unitario'].astype(str).str.replace(',', '.', regex=False)

df['precio_unitario'] = df['precio_unitario'].astype(float)

print(f"{df['precio_unitario'].dtype}")
df['precio_unitario']

float64


0       179000.0
1       211000.0
2       128000.0
3       133000.0
4        61000.0
          ...   
5995    180000.0
5996    130000.0
5997     77000.0
5998    126000.0
5999     99000.0
Name: precio_unitario, Length: 6000, dtype: float64

In [ ]:
# Limpiar tallas y descuentos
print(f"Tallas Vacias Antes: {df['talla'].isna().sum()}")
print(f"Descuentos Vacios Antes: {df['descuento'].isna().sum()}")

df['talla'] = df['talla'].fillna('Sin Talla').str.title().str.strip()
df['descuento'] = df['descuento'].fillna(0)

print(f"\nTallas Vacias despeus: {df['talla'].isna().sum()}")
print(f"\nDescuentos vacios despues: {df['descuento'].isna().sum()}")

Tallas Vacias Antes: 15
Descuentos Vacios Antes: 12
Tallas Vacias despeus: 0

Descuentos vacios despues: 0


In [5]:
# Limpiar texto en todas las columnas
columnas_texto = ['ciudad', 'departamento', 'nombre_tienda', 'nombre_producto', 'categoria',
                'canal', 'metodo_pago', 'temporada', 'genero_cliente', 'tipo_cliente', 'tipo_prenda']

for col in columnas_texto:
    df[col] = df[col].astype(str).str.title().str.strip()

In [6]:
# Convertir fechas

print(f"Tipo de fecha antes: {df['fecha_venta'].dtype}")

df['fecha_venta'] = pd.to_datetime(df['fecha_venta'], format="mixed", errors='coerce')

print(f"\nTipo de fecha despues: {df['fecha_venta'].dtype}")
print(f"Rango: {df['fecha_venta'].min()} - {df['fecha_venta'].max()}")

Tipo de fecha antes: str

Tipo de fecha despues: datetime64[us]
Rango: 2023-01-01 00:00:00 - 2025-12-31 00:00:00


In [ ]:
# Eliminar duplicados

filas_antes = df.shape[0]
df = df.drop_duplicates(subset=['id_venta', 'numero_linea'], keep='first')

filas_despues = df.shape[0]
print(f"Filas antes: {filas_antes},\nFilas despues: {filas_despues}\nFilas eliminadas: {filas_antes - filas_despues}")

Filas antes: 6000, Filas despues: 5993, Filas eliminadas: 7


In [ ]:
# Nulos parte 1

nulos = df.isnull().sum()
print(f"Nulos:\n{nulos[nulos > 0]}")

# Intentar encontrar al usuario que tiene nulos en la ciudad
clientes_nulos_ciudad = df[df['ciudad'].isna()]['id_cliente'].unique()
print(f"\nClientes con nulos en ciudad: {clientes_nulos_ciudad}")

historial_clientes = df[df['id_cliente'].isin(clientes_nulos_ciudad)]

print(historial_clientes[['id_cliente', 'ciudad', 'departamento', "nombre_tienda", "nombre_producto"]])



Nulos:
ciudad             3
nombre_producto    3
categoria          2
genero_cliente     2
metodo_pago        2
dtype: int64

Clientes con nulos en ciudad: <StringArray>
['C01353', 'C01498', 'C01386']
Length: 3, dtype: str
     id_cliente        ciudad departamento          nombre_tienda  \
945      C01353   Bucaramanga    Santander         Koaj La Quinta   
1383     C01498    Valledupar        Cesar           Koaj Mayales   
2061     C01498     Cartagena      Bolívar        Koaj Mall Plaza   
3364     C01498  Barranquilla    Atlántico  Koaj Portal Del Prado   
3460     C01353           NaN    Santander         Koaj La Quinta   
3521     C01498    Valledupar        Cesar           Koaj Mayales   
3634     C01498   Bucaramanga    Santander         Koaj La Quinta   
4159     C01498           NaN    Santander         Koaj La Quinta   
4298     C01386           NaN      Córdoba        Koaj Buenavista   
4866     C01498     Cartagena      Bolívar        Koaj Mall Plaza   
4884     C01386   

In [ ]:
# Llenar los nulos de ciudad de los clientes parte 2

df.loc[(df['id_cliente'] == 'C01353') & (df['nombre_tienda'] == 'Koaj La Quinta') & (df['ciudad'].isna()), 'ciudad'] = 'Bucaramanga'
df.loc[(df['id_cliente'] == 'C01498') & (df['nombre_tienda'] == 'Koaj La Quinta') & (df['ciudad'].isna()), 'ciudad'] = 'Bucaramanga'
df.loc[(df['id_cliente'] == 'C01386') & (df['nombre_tienda'] == 'Koaj Buenavista') & (df['ciudad'].isna()), 'ciudad'] = 'Córdoba'

clientes_nulos_ciudad = df[df['ciudad'].isna()]['id_cliente'].unique()
print(f"\nClientes con nulos en ciudad: {clientes_nulos_ciudad}")

historial_clientes = df[df['id_cliente'].isin(clientes_nulos_ciudad)]

print(historial_clientes[['id_cliente', 'ciudad', 'departamento', "nombre_tienda", "nombre_producto"]])


Clientes con nulos en ciudad: <StringArray>
[]
Length: 0, dtype: str
Empty DataFrame
Columns: [id_cliente, ciudad, departamento, nombre_tienda, nombre_producto]
Index: []


In [ ]:
# Nulos de nombre de producto parte 3

nulos = df.isnull().sum()
print(f"Nulos:\n{nulos[nulos > 0]}")

# Intentar encontrar la id del producto que tiene nulos
productos_nulos = df[df['nombre_producto'].isna()]['id_producto'].unique()
print(f"\nProductos con nulos en nombre: {productos_nulos}")

historial_productos = df[df['id_producto'].isin(productos_nulos)]

print(historial_productos[['id_producto', 'nombre_producto']])

df.loc[(df['id_producto'] == 'P0024') & (df['nombre_producto'].isna()), 'nombre_producto'] = 'Chaqueta Denim'
df.loc[(df['id_producto'] == 'P0013') & (df['nombre_producto'].isna()), 'nombre_producto'] = 'Bolso'
df.loc[(df['id_producto'] == 'P0006') & (df['nombre_producto'].isna()), 'nombre_producto'] = 'Cinturón'

Nulos:
categoria         2
genero_cliente    2
metodo_pago       2
dtype: int64

Productos con nulos en nombre: <StringArray>
[]
Length: 0, dtype: str
Empty DataFrame
Columns: [id_producto, nombre_producto]
Index: []


In [40]:
# Nulos de nombre de categoria parte 4

nulos = df.isnull().sum()
print(f"Nulos:\n{nulos[nulos > 0]}")

# Intentar encontrar la id del producto que tiene nulos
categorias_nulos = df[df['categoria'].isna()]['id_producto'].unique()
print(f"\nProductos con nulos en categoria: {categorias_nulos}")

historial_categorias = df[df['id_producto'].isin(categorias_nulos)]

print(historial_categorias[['id_producto', 'nombre_producto', 'categoria']])

df.loc[(df['id_producto'] == 'P0004') & (df['categoria'].isna()), 'categoria'] = 'Jeans'
df.loc[(df['id_producto'] == 'P0011') & (df['categoria'].isna()), 'categoria'] = 'Buzos'


Nulos:
genero_cliente    2
metodo_pago       2
dtype: int64

Productos con nulos en categoria: <StringArray>
[]
Length: 0, dtype: str
Empty DataFrame
Columns: [id_producto, nombre_producto, categoria]
Index: []


In [42]:
# Limpiar nulos en genero_cliente y metodo_pago parte 5

df['metodo_pago'] = df['metodo_pago'].fillna('Sin dato')
df['genero_cliente'] = df['genero_cliente'].fillna('Sin dato')

nulos = df.isnull().sum()
print(f"Nulos:\n{nulos[nulos > 0]}")

Nulos:
Series([], dtype: int64)


In [43]:
# Guardar el dataframe

columnas_finales = [
    'id_venta', 'numero_linea', 'fecha_venta',
    'id_cliente', 'genero_cliente', 'tipo_cliente',
    'id_producto', 'nombre_producto', 'categoria', 'tipo_prenda',
    'id_tienda', 'nombre_tienda', 'ciudad', 'departamento', 'empresa',
    'cantidad', 'precio_unitario', 'descuento', 'metodo_pago', 
    'canal', 'temporada', 'talla'
]

df_limpio = df[columnas_finales].copy()
df_limpio.to_csv('../data/processed/ventas_limpio.csv', index=False, encoding='utf-8')

print(f"Guardado: data/processed/ventas_limpio.csv")
print(f"Filas: {df_limpio.shape[0]}, Columnas: {df_limpio.shape[1]}")


Guardado: data/processed/ventas_limpio.csv
Filas: 5993, Columnas: 22
